# Time-Use Method Comparison

Compare district-level outputs across baseline, archetype assignment, and classifier workflows.

This notebook replaces the old duplicate archetype-only evaluation notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px


In [ ]:
DAY_TYPE = "weekday"
PROC_TIME_USE_DIR = PROJECT_ROOT / "data" / "processed" / "timeuse_profiles"

CANDIDATE_FILES = {
    "baseline": PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}.csv",
    "archetype": PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}_archetype.csv",
    "archetype_da50": PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}_archetype_da50.csv",
    "classifier": PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}_archetype_classifier.csv",
    "classifier_da50": PROC_TIME_USE_DIR / f"district_hourly_profiles_{DAY_TYPE}_archetype_classifier_da50.csv",
}


In [ ]:
frames = []
for method, path in CANDIDATE_FILES.items():
    if path.exists():
        df = pd.read_csv(path)
        df["method"] = method
        frames.append(df)

if not frames:
    raise FileNotFoundError("No district profile outputs were found for the configured DAY_TYPE.")

compare_df = pd.concat(frames, ignore_index=True)
compare_df["district_id"] = compare_df["district_id"].astype(str)

mean_profiles = (
    compare_df.groupby(["method", "hour"], as_index=False)[
        ["share_home_awake", "share_sleep", "share_away", "share_home_total"]
    ]
    .mean()
)
display(mean_profiles.head())


In [ ]:
plot_df = mean_profiles.melt(
    id_vars=["method", "hour"],
    value_vars=["share_home_awake", "share_sleep", "share_away", "share_home_total"],
    var_name="state",
    value_name="share",
)
fig = px.line(
    plot_df,
    x="hour",
    y="share",
    color="state",
    line_dash="method",
    title=f"Time-use method comparison ({DAY_TYPE})",
)
fig.show()

district_summary = compare_df.groupby(["method", "district_id"], as_index=False).agg(
    mean_home_total=("share_home_total", "mean"),
    max_home_total=("share_home_total", "max"),
    mean_away=("share_away", "mean"),
)
display(district_summary.head())
